# 📗 Module 2 - SQL ADVANCE (Intermediate to Advanced)

## Course Overview
This module builds on Module 1 foundations and introduces powerful SQL techniques for working with multiple tables, complex data analysis, and database optimization.

### What You'll Learn
* **JOINs** - Combine data from multiple tables
* **GROUP BY & HAVING** - Advanced aggregations and filtering
* **Subqueries** - Queries within queries
* **CTEs (Common Table Expressions)** - Readable complex queries
* **Window Functions** - Advanced analytics without grouping
* **Views** - Reusable query logic
* **Indexes** - Performance optimization
* **Set Operations** - UNION, INTERSECT, EXCEPT
* **Transactions** - Data consistency and integrity
* **Constraints** - Data validation rules
* **Advanced Functions** - String, date, JSON manipulation
* **Temporary Tables** - Intermediate result storage

**Prerequisites:** Module 1 - SQL BASIC

**Duration:** 8-12 hours

---

In [0]:
%sql
-- Set the default catalog and schema for this module
USE CATALOG workspace;
USE SCHEMA default;

-- Verify setup
SELECT 
  current_catalog() AS catalog,
  current_schema() AS schema,
  '✅ Ready to start Module 2!' AS status;

## 1️⃣ Table Relationships & Keys

Before diving into JOINs, understand how tables relate to each other.

### Types of Relationships

#### One-to-Many (1:N)
Most common relationship. One record in Table A relates to multiple records in Table B.
* Example: One customer can have many orders
* One department can have many employees

#### Many-to-Many (M:N)
Multiple records in Table A relate to multiple records in Table B.
* Requires a junction/bridge table
* Example: Students and Courses (one student takes many courses, one course has many students)

#### One-to-One (1:1)
One record in Table A relates to exactly one record in Table B.
* Less common
* Example: Employee and EmployeeDetails

### Foreign Keys
A foreign key in one table references the primary key in another table, establishing the relationship.

```sql
CREATE TABLE Orders (
  OrderID INT PRIMARY KEY,
  CustomerID INT,
  FOREIGN KEY (CustomerID) REFERENCES Customers(CustomerID)
);
```

## 2️⃣ INNER JOIN

Returns only rows where there is a match in BOTH tables.

### Syntax
```sql
SELECT columns
FROM table1
INNER JOIN table2
  ON table1.column = table2.column;
```

### Visual Representation
```
Table A         Table B
  ●             ●
    ●   [Match]  ●      ← INNER JOIN returns only matching rows
      ●         ●
```

### When to Use
* When you only want records that exist in both tables
* Most common type of JOIN
* Example: Get orders with customer information (only show orders that have a customer)

In [0]:
%sql
-- Basic INNER JOIN: Customers and their orders
SELECT 
  c.CustomerName,
  c.City,
  o.OrderID,
  o.OrderDate,
  o.Amount,
  o.Status
FROM Customers c
INNER JOIN Orders o
  ON c.CustomerID = o.CustomerID;

-- Multiple column selection with table aliases
-- SELECT 
--   c.CustomerName AS Customer,
--   COUNT(o.OrderID) AS TotalOrders,
--   SUM(o.Amount) AS TotalSpent
-- FROM Customers c
-- INNER JOIN Orders o ON c.CustomerID = o.CustomerID
-- GROUP BY c.CustomerID, c.CustomerName;

## 3️⃣ LEFT JOIN (LEFT OUTER JOIN)

Returns ALL rows from the left table, and matching rows from the right table. If no match, NULL values for right table columns.

### Syntax
```sql
SELECT columns
FROM table1
LEFT JOIN table2
  ON table1.column = table2.column;
```

### Visual Representation
```
Table A (Left)  Table B (Right)
  ●                ○
    ●   [Match]   ●
      ●            ○
```
LEFT JOIN returns all from A (● and ●), with B data where available

### When to Use
* Find all records from primary table, whether or not they have related records
* Example: Show all customers, including those who haven't placed orders
* Find orphaned records (customers with no orders)

In [0]:
%sql
-- LEFT JOIN: All customers, with their orders (if any)
SELECT 
  c.CustomerName,
  c.City,
  o.OrderID,
  o.OrderDate,
  o.Amount
FROM Customers c
LEFT JOIN Orders o
  ON c.CustomerID = o.CustomerID;

-- Find customers who haven't placed any orders
-- SELECT 
--   c.CustomerName,
--   c.City,
--   c.JoinDate
-- FROM Customers c
-- LEFT JOIN Orders o ON c.CustomerID = o.CustomerID
-- WHERE o.OrderID IS NULL;

## 4️⃣ RIGHT JOIN & FULL OUTER JOIN

### RIGHT JOIN (RIGHT OUTER JOIN)
Returns ALL rows from the right table, and matching rows from the left table.

```sql
SELECT columns
FROM table1
RIGHT JOIN table2
  ON table1.column = table2.column;
```

**Note:** RIGHT JOIN is less commonly used. You can achieve the same result by swapping tables and using LEFT JOIN.

### FULL OUTER JOIN
Returns ALL rows from BOTH tables. Where there's no match, NULL values appear.

```sql
SELECT columns
FROM table1
FULL OUTER JOIN table2
  ON table1.column = table2.column;
```

### When to Use
* **RIGHT JOIN**: Rarely used (prefer LEFT JOIN with swapped tables)
* **FULL OUTER JOIN**: Find all records from both tables, including unmatched
  - Example: All customers and all orders, showing orphaned records from both sides

In [0]:
%sql
-- RIGHT JOIN: All orders with customer info (if available)
SELECT 
  c.CustomerName,
  o.OrderID,
  o.OrderDate,
  o.Amount
FROM Customers c
RIGHT JOIN Orders o
  ON c.CustomerID = o.CustomerID;

-- FULL OUTER JOIN: All customers and all orders
-- Shows matched and unmatched from both sides
-- SELECT 
--   c.CustomerName,
--   c.City,
--   o.OrderID,
--   o.Amount
-- FROM Customers c
-- FULL OUTER JOIN Orders o
--   ON c.CustomerID = o.CustomerID;

## 5️⃣ CROSS JOIN & Self JOIN

### CROSS JOIN (Cartesian Product)
Returns ALL possible combinations of rows from both tables.

```sql
SELECT * FROM table1 CROSS JOIN table2;
```

**Warning:** Can produce very large result sets!
* Table A has 10 rows, Table B has 20 rows → Result has 200 rows (10 × 20)

**Use Cases:**
* Generating all combinations (e.g., all sizes for all products)
* Creating test data
* Calendar tables

### Self JOIN
Joining a table to itself. Useful for hierarchical data or comparing rows within same table.

```sql
SELECT e1.Name AS Employee, e2.Name AS Manager
FROM Employees e1
INNER JOIN Employees e2
  ON e1.ManagerID = e2.EmployeeID;
```

In [0]:
%sql
-- Self JOIN: Find employees and their managers
SELECT 
  concat(e.FirstName, ' ', e.LastName) AS Employee,
  concat(m.FirstName, ' ', m.LastName) AS Manager,
  e.Department
FROM Employees e
LEFT JOIN Employees m
  ON e.ManagerID = m.EmployeeID;

-- Self JOIN: Find colleagues (employees with same manager)
-- SELECT 
--   e1.FirstName AS Employee1,
--   e2.FirstName AS Employee2,
--   e1.Department
-- FROM Employees e1
-- INNER JOIN Employees e2
--   ON e1.ManagerID = e2.ManagerID
--   AND e1.EmployeeID < e2.EmployeeID  -- Avoid duplicates
-- WHERE e1.ManagerID IS NOT NULL;

## 6️⃣ Multiple JOINs

You can join more than two tables in a single query.

### Syntax
```sql
SELECT columns
FROM table1
JOIN table2 ON table1.key = table2.key
JOIN table3 ON table2.key = table3.key
JOIN table4 ON table3.key = table4.key;
```

### Best Practices
* Use meaningful table aliases (not just a, b, c)
* Build joins logically (start with main table)
* Consider join order for performance
* Use appropriate join types (INNER vs LEFT)

In [0]:
%sql
-- Join Employees, Departments (if exists), and their Manager info
SELECT 
  concat(e.FirstName, ' ', e.LastName) AS EmployeeName,
  e.Department,
  e.Salary,
  concat(m.FirstName, ' ', m.LastName) AS ManagerName,
  m.Department AS ManagerDepartment
FROM Employees e
LEFT JOIN Employees m ON e.ManagerID = m.EmployeeID
WHERE e.Department = 'IT';

-- Complex: Customers, Orders, and Order summary
-- SELECT 
--   c.CustomerName,
--   c.City,
--   o.OrderID,
--   o.OrderDate,
--   o.Amount,
--   o.Status
-- FROM Customers c
-- LEFT JOIN Orders o ON c.CustomerID = o.CustomerID
-- WHERE c.City = 'New York' OR o.Status = 'Completed';

## 7️⃣ GROUP BY - Aggregating Data

GROUP BY groups rows with same values into summary rows.

### Syntax
```sql
SELECT column1, aggregate_function(column2)
FROM table_name
GROUP BY column1;
```

### Aggregate Functions
* `COUNT(*)` - Count all rows
* `COUNT(column)` - Count non-NULL values
* `SUM(column)` - Total of values
* `AVG(column)` - Average of values
* `MIN(column)` - Minimum value
* `MAX(column)` - Maximum value

### Rules
* Every non-aggregated column in SELECT must be in GROUP BY
* GROUP BY happens before ORDER BY
* Can group by multiple columns

In [0]:
%sql
-- Count employees per department
SELECT 
  Department,
  COUNT(*) AS EmployeeCount,
  AVG(Salary) AS AvgSalary,
  MIN(Salary) AS MinSalary,
  MAX(Salary) AS MaxSalary
FROM Employees
GROUP BY Department
ORDER BY EmployeeCount DESC;

-- Group by multiple columns: Department and Manager
-- SELECT 
--   Department,
--   ManagerID,
--   COUNT(*) AS TeamSize,
--   AVG(Salary) AS AvgTeamSalary
-- FROM Employees
-- GROUP BY Department, ManagerID;

## 8️⃣ HAVING - Filtering Groups

HAVING filters groups AFTER aggregation (WHERE filters rows BEFORE aggregation).

### Syntax
```sql
SELECT column1, aggregate_function(column2)
FROM table_name
GROUP BY column1
HAVING aggregate_function(column2) condition;
```

### WHERE vs HAVING
* **WHERE**: Filters individual rows BEFORE grouping
  - Use for filtering on non-aggregated columns
  - Example: `WHERE Salary > 50000`
  
* **HAVING**: Filters groups AFTER aggregation
  - Use for filtering on aggregated values
  - Example: `HAVING AVG(Salary) > 70000`

### Query Execution Order
```
FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY → LIMIT
```

In [0]:
%sql
-- Departments with average salary > 70000
SELECT 
  Department,
  COUNT(*) AS EmployeeCount,
  AVG(Salary) AS AvgSalary,
  SUM(Salary) AS TotalPayroll
FROM Employees
GROUP BY Department
HAVING AVG(Salary) > 70000
ORDER BY AvgSalary DESC;

-- Departments with more than 2 employees
-- SELECT 
--   Department,
--   COUNT(*) AS EmployeeCount
-- FROM Employees
-- GROUP BY Department
-- HAVING COUNT(*) > 2;

-- Combine WHERE and HAVING
-- SELECT 
--   Department,
--   AVG(Salary) AS AvgSalary
-- FROM Employees
-- WHERE HireDate >= '2020-01-01'  -- Filter rows first
-- GROUP BY Department
-- HAVING AVG(Salary) > 65000;      -- Filter groups after

## 9️⃣ Subqueries (Nested Queries)

A subquery is a query nested inside another query.

### Types of Subqueries

#### 1. Scalar Subquery
Returns a single value (one row, one column)
```sql
SELECT * FROM Employees WHERE Salary > (SELECT AVG(Salary) FROM Employees);
```

#### 2. Row Subquery
Returns a single row (multiple columns)

#### 3. Table Subquery
Returns multiple rows (used with IN, EXISTS, ANY, ALL)
```sql
SELECT * FROM Employees WHERE DepartmentID IN (SELECT DepartmentID FROM Departments WHERE Budget > 100000);
```

### Where to Use Subqueries
* **SELECT clause** - Calculate values
* **FROM clause** - Derived tables
* **WHERE clause** - Filter conditions
* **HAVING clause** - Filter groups

In [0]:
%sql
-- Subquery in WHERE: Find employees earning above average
SELECT 
  FirstName,
  LastName,
  Salary,
  Department
FROM Employees
WHERE Salary > (SELECT AVG(Salary) FROM Employees);

-- Subquery with IN: Employees in high-performing departments
-- SELECT FirstName, LastName, Department
-- FROM Employees
-- WHERE Department IN (
--   SELECT Department 
--   FROM Employees 
--   GROUP BY Department 
--   HAVING AVG(Salary) > 70000
-- );

-- Subquery in SELECT: Compare each salary to department average
-- SELECT 
--   FirstName,
--   Salary,
--   Department,
--   (SELECT AVG(Salary) FROM Employees e2 WHERE e2.Department = e1.Department) AS DeptAvg
-- FROM Employees e1;

## 🔟 Correlated Subqueries

A correlated subquery references columns from the outer query. It executes once for each row of the outer query.

### Syntax
```sql
SELECT outer.*
FROM table1 outer
WHERE column > (
  SELECT aggregate_function(column)
  FROM table1 inner
  WHERE inner.group_column = outer.group_column  -- Correlation
);
```

### Characteristics
* **Dependent**: Inner query depends on outer query
* **Performance**: Can be slower (executes multiple times)
* **Use Cases**: Row-by-row comparisons within groups

### EXISTS vs IN
* `EXISTS`: Checks if subquery returns any rows (faster for large datasets)
* `IN`: Checks if value matches any value in subquery

In [0]:
%sql
-- Employees earning above their department average
SELECT 
  e1.FirstName,
  e1.LastName,
  e1.Salary,
  e1.Department
FROM Employees e1
WHERE e1.Salary > (
  SELECT AVG(e2.Salary)
  FROM Employees e2
  WHERE e2.Department = e1.Department
);

-- Using EXISTS: Customers who have placed orders
-- SELECT c.CustomerName, c.City
-- FROM Customers c
-- WHERE EXISTS (
--   SELECT 1 FROM Orders o WHERE o.CustomerID = c.CustomerID
-- );

-- Using NOT EXISTS: Customers who haven't placed orders
-- SELECT c.CustomerName, c.City
-- FROM Customers c
-- WHERE NOT EXISTS (
--   SELECT 1 FROM Orders o WHERE o.CustomerID = c.CustomerID
-- );

## 1️⃣1️⃣ CTEs (Common Table Expressions)

CTEs create temporary named result sets that exist only during query execution.

### Syntax
```sql
WITH cte_name AS (
  SELECT columns FROM table WHERE condition
)
SELECT * FROM cte_name;
```

### Benefits
* **Readability**: Break complex queries into logical parts
* **Reusability**: Reference CTE multiple times in same query
* **Recursion**: CTEs can be recursive (self-referencing)
* **Maintainability**: Easier to understand and modify

### Multiple CTEs
```sql
WITH 
  cte1 AS (SELECT ...),
  cte2 AS (SELECT ...)
SELECT * FROM cte1 JOIN cte2 ON ...;
```

In [0]:
%sql
-- Basic CTE: High-earning employees
WITH HighEarners AS (
  SELECT FirstName, LastName, Salary, Department
  FROM Employees
  WHERE Salary > 70000
)
SELECT 
  Department,
  COUNT(*) AS HighEarnerCount,
  AVG(Salary) AS AvgSalary
FROM HighEarners
GROUP BY Department;

-- Multiple CTEs: Department stats and employee comparison
-- WITH 
--   DeptStats AS (
--     SELECT 
--       Department,
--       AVG(Salary) AS AvgSalary,
--       COUNT(*) AS EmpCount
--     FROM Employees
--     GROUP BY Department
--   ),
--   HighPerformers AS (
--     SELECT FirstName, LastName, Salary, Department
--     FROM Employees
--     WHERE Salary > 75000
--   )
-- SELECT 
--   h.FirstName,
--   h.Salary,
--   h.Department,
--   d.AvgSalary AS DeptAvg,
--   d.EmpCount AS DeptSize
-- FROM HighPerformers h
-- JOIN DeptStats d ON h.Department = d.Department;

## 1️⃣2️⃣ Recursive CTEs

Recursive CTEs reference themselves, useful for hierarchical data (org charts, bill of materials, folder structures).

### Syntax
```sql
WITH RECURSIVE cte_name AS (
  -- Anchor member (base case)
  SELECT columns WHERE base_condition
  
  UNION ALL
  
  -- Recursive member (references CTE itself)
  SELECT columns FROM table
  JOIN cte_name ON join_condition
)
SELECT * FROM cte_name;
```

### How It Works
1. **Anchor**: Base rows (starting point)
2. **Recursion**: Joins result to itself repeatedly
3. **Termination**: Stops when no new rows are produced

### Common Uses
* Organization hierarchies (manager-employee trees)
* Bill of materials (part-subpart relationships)
* Graph traversal
* Date/number sequences

In [0]:
%sql
-- Recursive CTE: Employee hierarchy (org chart)
WITH RECURSIVE OrgChart AS (
  -- Anchor: Top-level managers (no manager)
  SELECT 
    EmployeeID,
    FirstName,
    LastName,
    ManagerID,
    1 AS Level,
    CAST(concat(FirstName, ' ', LastName) AS VARCHAR(1000)) AS OrgPath
  FROM Employees
  WHERE ManagerID IS NULL
  
  UNION ALL
  
  -- Recursive: Employees under managers
  SELECT 
    e.EmployeeID,
    e.FirstName,
    e.LastName,
    e.ManagerID,
    oc.Level + 1,
    concat(oc.OrgPath, ' -> ', e.FirstName, ' ', e.LastName)
  FROM Employees e
  INNER JOIN OrgChart oc ON e.ManagerID = oc.EmployeeID
)
SELECT 
  EmployeeID,
  concat(FirstName, ' ', LastName) AS EmployeeName,
  Level,
  OrgPath
FROM OrgChart
ORDER BY Level, EmployeeID;

-- Generate number sequence using recursive CTE
-- WITH RECURSIVE Numbers AS (
--   SELECT 1 AS n
--   UNION ALL
--   SELECT n + 1 FROM Numbers WHERE n < 10
-- )
-- SELECT n FROM Numbers;

## 1️⃣3️⃣ Window Functions (Analytic Functions)

Window functions perform calculations across a set of rows related to the current row WITHOUT collapsing rows like GROUP BY.

### Key Difference from GROUP BY
* **GROUP BY**: Collapses multiple rows into one summary row
* **Window Functions**: Keeps all rows, adds calculated column

### Syntax
```sql
function_name(column) OVER (
  [PARTITION BY column]
  [ORDER BY column]
  [ROWS/RANGE frame_specification]
)
```

### Components
* **Function**: The calculation to perform
* **OVER**: Defines the window
* **PARTITION BY**: Groups rows (like GROUP BY, but doesn't collapse)
* **ORDER BY**: Sorts rows within each partition
* **Frame**: Subset of partition for calculation

### Types of Window Functions

#### 1. Ranking Functions
* `ROW_NUMBER()` - Unique sequential number
* `RANK()` - Rank with gaps for ties
* `DENSE_RANK()` - Rank without gaps
* `NTILE(n)` - Divides rows into n groups

#### 2. Aggregate Window Functions
* `SUM()`, `AVG()`, `COUNT()`, `MIN()`, `MAX()`

#### 3. Value Functions
* `LAG()` - Access previous row
* `LEAD()` - Access next row
* `FIRST_VALUE()` - First value in window
* `LAST_VALUE()` - Last value in window

In [0]:
%sql
-- Ranking employees by salary
SELECT 
  FirstName,
  LastName,
  Department,
  Salary,
  ROW_NUMBER() OVER (ORDER BY Salary DESC) AS RowNum,
  RANK() OVER (ORDER BY Salary DESC) AS Rank,
  DENSE_RANK() OVER (ORDER BY Salary DESC) AS DenseRank
FROM Employees;

-- Rank within each department
-- SELECT 
--   FirstName,
--   Department,
--   Salary,
--   RANK() OVER (PARTITION BY Department ORDER BY Salary DESC) AS DeptRank
-- FROM Employees;

-- NTILE: Divide employees into 4 salary quartiles
-- SELECT 
--   FirstName,
--   Salary,
--   NTILE(4) OVER (ORDER BY Salary) AS SalaryQuartile
-- FROM Employees;

In [0]:
%sql
-- Running total of salaries
SELECT 
  FirstName,
  Department,
  Salary,
  SUM(Salary) OVER (ORDER BY EmployeeID) AS RunningTotal,
  AVG(Salary) OVER (PARTITION BY Department) AS DeptAvgSalary,
  Salary - AVG(Salary) OVER (PARTITION BY Department) AS DiffFromDeptAvg
FROM Employees;

-- Moving average (last 3 rows)
-- SELECT 
--   FirstName,
--   Salary,
--   AVG(Salary) OVER (
--     ORDER BY EmployeeID
--     ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
--   ) AS MovingAvg3
-- FROM Employees;

## 1️⃣4️⃣ Window Functions - LAG and LEAD

### LAG() - Access Previous Row
Returns value from a previous row in the result set.

```sql
LAG(column, offset, default) OVER (ORDER BY column)
```

### LEAD() - Access Next Row
Returns value from a subsequent row in the result set.

```sql
LEAD(column, offset, default) OVER (ORDER BY column)
```

### Parameters
* **column**: The column to retrieve
* **offset**: How many rows back/forward (default: 1)
* **default**: Value to return if no row exists (default: NULL)

### Use Cases
* Compare current row with previous/next
* Calculate deltas (difference from previous value)
* Find trends in time-series data
* Identify consecutive events

In [0]:
%sql
-- Compare each employee's salary to previous employee
SELECT 
  FirstName,
  Department,
  Salary,
  LAG(Salary) OVER (ORDER BY Salary) AS PreviousSalary,
  Salary - LAG(Salary) OVER (ORDER BY Salary) AS SalaryDiff,
  LEAD(Salary) OVER (ORDER BY Salary) AS NextSalary
FROM Employees;

-- Year-over-year comparison (if we had yearly data)
-- SELECT 
--   OrderDate,
--   Amount,
--   LAG(Amount, 1) OVER (ORDER BY OrderDate) AS PrevOrderAmount,
--   Amount - LAG(Amount, 1) OVER (ORDER BY OrderDate) AS AmountChange
-- FROM Orders;

## 1️⃣5️⃣ Views - Reusable Queries

A view is a virtual table based on a SELECT query. It doesn't store data itself, but provides a saved query definition.

### Syntax
```sql
CREATE VIEW view_name AS
SELECT columns FROM table WHERE condition;

-- Use the view
SELECT * FROM view_name;
```

### Benefits
* **Simplification**: Hide complex JOINs and logic
* **Security**: Limit access to specific columns/rows
* **Consistency**: Ensure same logic used across queries
* **Abstraction**: Change underlying tables without affecting queries

### View Operations
```sql
-- Create or replace view
CREATE OR REPLACE VIEW view_name AS SELECT ...;

-- Drop view
DROP VIEW view_name;

-- Show view definition
SHOW CREATE VIEW view_name;
```

### Materialized Views (Advanced)
Stores query results physically for faster access (covered in Module 3)

In [0]:
%sql
-- Create a view for high-earning employees
CREATE OR REPLACE VIEW HighEarners AS
SELECT 
  FirstName,
  LastName,
  Department,
  Salary,
  Email
FROM Employees
WHERE Salary > 70000;

-- Use the view
SELECT * FROM HighEarners;

-- View with JOINs: Customer order summary
-- CREATE OR REPLACE VIEW CustomerOrderSummary AS
-- SELECT 
--   c.CustomerID,
--   c.CustomerName,
--   c.City,
--   COUNT(o.OrderID) AS TotalOrders,
--   COALESCE(SUM(o.Amount), 0) AS TotalSpent,
--   MAX(o.OrderDate) AS LastOrderDate
-- FROM Customers c
-- LEFT JOIN Orders o ON c.CustomerID = o.CustomerID
-- GROUP BY c.CustomerID, c.CustomerName, c.City;

-- SELECT * FROM CustomerOrderSummary WHERE TotalOrders > 0;

## 1️⃣6️⃣ Indexes - Performance Optimization

Indexes speed up data retrieval at the cost of slower writes and additional storage.

### What is an Index?
A data structure (like a book index) that helps find rows quickly without scanning the entire table.

### Types of Indexes

#### 1. Clustered Index
* Determines physical order of data in table
* Only ONE per table
* Automatically created on PRIMARY KEY

#### 2. Non-Clustered Index
* Separate structure pointing to data
* Multiple per table
* Like book index with page numbers

#### 3. Unique Index
* Ensures no duplicate values
* Automatically created on UNIQUE constraint

#### 4. Composite Index
* Index on multiple columns
* Order matters: (A, B) ≠ (B, A)

### Syntax
```sql
-- Create index
CREATE INDEX idx_name ON table(column);

-- Create composite index
CREATE INDEX idx_name ON table(col1, col2);

-- Create unique index
CREATE UNIQUE INDEX idx_name ON table(column);

-- Drop index
DROP INDEX idx_name;
```

### When to Use Indexes
✅ **Create indexes on**:
* Columns frequently in WHERE clauses
* JOIN columns
* ORDER BY columns
* Foreign key columns

❌ **Avoid indexes on**:
* Small tables
* Columns with many duplicates
* Columns frequently updated
* Rarely queried columns

In [0]:
%sql
-- Create index on Department (frequently filtered)
CREATE INDEX idx_employees_dept ON Employees(Department);

-- Create composite index on Department and Salary
-- CREATE INDEX idx_employees_dept_salary ON Employees(Department, Salary);

-- Create unique index on Email
-- CREATE UNIQUE INDEX idx_employees_email ON Employees(Email);

-- Show indexes on a table (syntax varies by database)
-- SHOW INDEXES FROM Employees;

-- Drop an index
-- DROP INDEX idx_employees_dept;

## 1️⃣7️⃣ Set Operations (UNION, INTERSECT, EXCEPT)

Combine results from multiple SELECT queries.

### Requirements
* Same number of columns
* Compatible data types
* Column names from first query used

### UNION
Combines results, removes duplicates.
```sql
SELECT columns FROM table1
UNION
SELECT columns FROM table2;
```

### UNION ALL
Combines results, keeps duplicates (faster).
```sql
SELECT columns FROM table1
UNION ALL
SELECT columns FROM table2;
```

### INTERSECT
Returns only rows present in BOTH queries.
```sql
SELECT columns FROM table1
INTERSECT
SELECT columns FROM table2;
```

### EXCEPT (or MINUS)
Returns rows from first query NOT in second query.
```sql
SELECT columns FROM table1
EXCEPT
SELECT columns FROM table2;
```

### Performance Tips
* Use UNION ALL when duplicates don't matter (faster)
* INTERSECT/EXCEPT can often be rewritten with JOINs

In [0]:
%sql
-- UNION: All unique cities from Customers and Suppliers
SELECT City FROM Customers
UNION
SELECT City FROM Suppliers;

-- UNION ALL: All cities including duplicates
-- SELECT City FROM Customers
-- UNION ALL
-- SELECT City FROM Suppliers;

-- INTERSECT: Cities that have both customers and suppliers
-- SELECT City FROM Customers
-- INTERSECT
-- SELECT City FROM Suppliers;

-- EXCEPT: Cities with customers but no suppliers
-- SELECT City FROM Customers
-- EXCEPT
-- SELECT City FROM Suppliers;

-- Complex: Combine different datasets
-- SELECT FirstName AS Name, 'Employee' AS Type FROM Employees
-- UNION
-- SELECT CustomerName AS Name, 'Customer' AS Type FROM Customers
-- ORDER BY Type, Name;

## 1️⃣8️⃣ Transactions - Data Integrity

A transaction is a sequence of operations performed as a single logical unit of work. All operations must succeed, or all must fail.

### ACID Properties
* **Atomicity**: All or nothing (transaction either completes fully or not at all)
* **Consistency**: Database remains in valid state
* **Isolation**: Concurrent transactions don't interfere
* **Durability**: Committed changes are permanent

### Transaction Commands
```sql
-- Start transaction
BEGIN TRANSACTION;  -- or START TRANSACTION;

-- Perform operations
UPDATE accounts SET balance = balance - 100 WHERE id = 1;
UPDATE accounts SET balance = balance + 100 WHERE id = 2;

-- Commit (save changes)
COMMIT;

-- Rollback (undo changes)
ROLLBACK;
```

### Savepoints
Create intermediate points to rollback to.
```sql
BEGIN TRANSACTION;
  UPDATE table1 ...;
  SAVEPOINT sp1;
  UPDATE table2 ...;
  ROLLBACK TO sp1;  -- Undo only table2 update
COMMIT;
```

### When to Use Transactions
* Financial operations (transfers, payments)
* Multi-step operations that must complete together
* Ensuring data consistency
* Coordinating related updates across tables

In [0]:
%sql
-- Transaction: Transfer money between accounts
BEGIN TRANSACTION;

-- Withdraw from Account 1
UPDATE Accounts 
SET Balance = Balance - 500 
WHERE AccountID = 1;

-- Deposit to Account 2
UPDATE Accounts 
SET Balance = Balance + 500 
WHERE AccountID = 2;

-- If both succeed, commit
COMMIT;

-- If error occurs, rollback
-- ROLLBACK;

-- Transaction with savepoint
-- BEGIN TRANSACTION;
--   INSERT INTO Orders (OrderID, CustomerID, Amount) VALUES (1006, 1, 250);
--   SAVEPOINT after_order;
--   
--   UPDATE Customers SET LastOrderDate = CURRENT_DATE WHERE CustomerID = 1;
--   
--   -- If update fails, rollback to savepoint
--   -- ROLLBACK TO after_order;
-- COMMIT;

## 1️⃣9️⃣ Temporary Tables

Temporary tables store intermediate results during a session and are automatically dropped when the session ends.

### Types

#### 1. Local Temporary Tables
Visible only to current session.
```sql
CREATE TEMPORARY TABLE temp_table (
  column1 datatype,
  column2 datatype
);
```

#### 2. Global Temporary Tables (SQL Server)
Visible to all sessions.
```sql
CREATE TABLE ##global_temp (...);  -- Double ## in SQL Server
```

### When to Use
* Break complex queries into steps
* Store intermediate results
* Improve performance for complex calculations
* ETL processes
* Reporting with multiple data transformations

### Temporary Tables vs CTEs vs Views
* **Temp Tables**: Physical storage, can be indexed, persists in session
* **CTEs**: Query-scoped, no physical storage, cleaner syntax
* **Views**: Saved definition, no physical storage, reusable across sessions

### Best Practices
* Use meaningful names
* Clean up explicitly if needed (though auto-dropped)
* Consider CTEs for simpler cases

In [0]:
%sql
-- Create temporary table for analysis
CREATE TEMPORARY TABLE TempHighEarners (
  EmployeeID INT,
  FullName VARCHAR(100),
  Department VARCHAR(50),
  Salary DECIMAL(10,2)
);

-- Insert data into temp table
INSERT INTO TempHighEarners
SELECT 
  EmployeeID,
  FirstName + ' ' + LastName AS FullName,
  Department,
  Salary
FROM Employees
WHERE Salary > 70000;

-- Query the temp table
SELECT * FROM TempHighEarners;

-- Use temp table in analysis
-- SELECT 
--   Department,
--   COUNT(*) AS HighEarnerCount,
--   AVG(Salary) AS AvgSalary
-- FROM TempHighEarners
-- GROUP BY Department;

-- Drop temp table (optional - auto-dropped at session end)
-- DROP TABLE TempHighEarners;

## 🎯 Module 2 Practice Exercises

Reinforce your learning with these comprehensive exercises.

### Exercise Set 1: JOINs
1. List all customers with their order count (include customers with 0 orders)
2. Find employees and their managers' names
3. Show all orders with customer names, sorted by order date
4. Find customers who have placed more than 2 orders

### Exercise Set 2: Aggregations & Window Functions
1. Find departments where average salary exceeds $70,000
2. Rank employees by salary within each department
3. Calculate running total of order amounts
4. Show each employee's salary vs department average

### Exercise Set 3: Subqueries & CTEs
1. Find employees earning more than the company average
2. Use CTE to find top 3 departments by average salary
3. Create a recursive CTE showing employee hierarchy

### Exercise Set 4: Advanced Operations
1. Create a view showing customer order summaries
2. Use UNION to list all unique cities from Customers and Suppliers
3. Create a temporary table for high-earning employees analysis

In [0]:
%sql
-- Use this cell to practice your SQL queries

-- Example Solution:
SELECT 
  c.CustomerName,
  COUNT(o.OrderID) AS OrderCount
FROM Customers c
LEFT JOIN Orders o ON c.CustomerID = o.CustomerID
GROUP BY c.CustomerID, c.CustomerName;

-- Your solutions here:



## 🎓 Module 2 Summary

Congratulations! You've mastered SQL ADVANCE techniques.

### ✅ Key Skills Acquired
* **JOINs** - INNER, LEFT, RIGHT, FULL OUTER, CROSS, Self
* **GROUP BY & HAVING** - Advanced aggregations
* **Subqueries & CTEs** - Including recursive CTEs
* **Window Functions** - Ranking, aggregates, LAG/LEAD
* **Views** - Reusable query logic
* **Indexes** - Performance optimization
* **Set Operations** - UNION, INTERSECT, EXCEPT
* **Transactions** - Data integrity & ACID
* **Temporary Tables** - Session storage

### 🚀 Next: Module 3 - SQL EXPERT

Master enterprise-level SQL with:
* Advanced database design
* Performance tuning
* Stored procedures & triggers
* Security & compliance
* High availability

**Practice these concepts thoroughly before advancing!**